In [ ]:
######################################## TRANSFER LEARNING #############################################

In [ ]:
import joblib, pandas as pd, numpy as np
from pathlib import Path

# Paths from your baseline step
BASELINE_CKPT      = Path("artifacts_medium_cv/final_fold_models/fold_1_best.pth")  # change with the correct model

# Low-range data
LOW_DF_PATH = "data/low_range/low_train.csv" # fix file path
TARGET_COL  = "MP"

# Load low-MW dataset - called the scaled version
df_low = pd.read_csv(LOW_DF_PATH)


In [ ]:
def set_freeze_mode(model, freeze_level=0):
    """
    freeze_level:
        0 = train all layers
        1 = freeze first hidden block
        2 = freeze first two hidden blocks
        3 = freeze first three hidden blocks (if present)
    """
    block_size = 4  # [Linear, BatchNorm, ReLU, Dropout] per hidden layer

    # Unfreeze everything first
    for p in model.parameters():
        p.requires_grad = True

    if freeze_level == 0:
        print("Freeze Level 0: all layers trainable")
        return

    # How many blocks actually exist?
    n_blocks_total = len(model.network) // block_size  # e.g., 3 blocks for [256,128,64]
    n_blocks = min(freeze_level, n_blocks_total)
    print(f"Freeze Level {freeze_level}: freezing {n_blocks} block(s)")

    for b in range(n_blocks):
        start = b * block_size  # 0, 4, 8, ...
        # Freeze Linear + BatchNorm in this block
        for i in range(start, start + 2):  # [Linear, BatchNorm]
            layer = model.network[i]
            for p in layer.parameters():
                p.requires_grad = False
            layer.eval()  # especially important for BatchNorm


In [ ]:
import numpy as np
from pathlib import Path
import torch
from sklearn.metrics import root_mean_squared_error, r2_score


def save_checkpoint(model, optimizer, epoch, train_loss, val_loss, y_train, y_val, val_loader, 
                   fold_idx, checkpoint_dir, checkpoint_tracking, is_final=False):
        
        
        # Calculate val predictions
        model.eval()
        all_preds = []
        with torch.no_grad():
            for xb, _ in val_loader:
                preds = model(xb).cpu().numpy()
                all_preds.append(preds)
        preds_val = np.concatenate(all_preds)
        
        # Calculate performance metrics from val predictions
        checkpoint_rmse = root_mean_squared_error(y_val, preds_val)
        checkpoint_r2 = r2_score(y_val, preds_val)
        checkpoint_q2 = 1 - np.sum((y_val - preds_val)**2) / np.sum((y_val - y_train.mean())**2)
        
        # Create checkpoint filename
        if is_final:
            checkpoint_filename = f"checkpoint_epoch_{epoch:03d}_final.pt"
        else:
            checkpoint_filename = f"checkpoint_epoch_{epoch:03d}.pt"
        
        checkpoint_path_full = Path(checkpoint_dir) / checkpoint_filename
        
        # Save the checkpoint
        checkpoint_data = {'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss, 'val_loss': val_loss, 'rmse': checkpoint_rmse, 'r2': checkpoint_r2, 'q2': checkpoint_q2,
            'fold_idx': fold_idx, 'is_final': is_final}
        torch.save(checkpoint_data, checkpoint_path_full)
        
        # Record info for spreadsheet
        checkpoint_info = {'Fold': fold_idx, 'Epoch': epoch, 'Checkpoint_Filename': checkpoint_filename, 'Checkpoint_Path': str(checkpoint_path_full),
            'Train_Loss': round(train_loss, 6), 'Val_Loss': round(val_loss, 6), 'RMSE': round(checkpoint_rmse, 6), 'R2': round(checkpoint_r2, 6), 
            'Q2': round(checkpoint_q2, 6), 'Is_Final': is_final}
        checkpoint_tracking.append(checkpoint_info)
        
        checkpoint_type = "Final" if is_final else "Regular"
        print(f"[Fold {fold_idx}] {checkpoint_type} checkpoint saved at epoch {epoch} - RMSE: {checkpoint_rmse:.4f}")
        return True
        

In [ ]:
# since RMSE Loss is not in PyTorch, we define it here using MSELoss

class RMSELoss(nn.Module):
    def __init__(self, eps=1e-8):  

        super().__init__()
        self.mse = nn.MSELoss()
        self.eps = eps      # eps: a small constant to avoid sqrt(0) or division by zero;  to prevent potential numerical instability or "division by zero" like issues if the MSE happens to be exactly zero 

    def forward(self, y_pred, y_true):
        mse = self.mse(y_pred, y_true)
        rmse = torch.sqrt(mse + self.eps)
        return rmse
    

#Source: https://stackoverflow.com/questions/71998978/early-stopping-in-pytorch

# Early Stopping Based on Validation Loss
class EarlyStopper:

    # If the val loss has not been improved (i.e. stayed the same or got worse) for 20 epochs in a row, the training of the model is stopped.
    def __init__(self, patience=30, min_delta=0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.counter = 0
        self.best_loss = float('inf')
        self.stop = False
        self.stop_epoch = None  # remember which epoch we stopped on (for plotting)

    def early_stop(self, val_loss, epoch=None):

        #For each epoch, checks if the validation loss has improved, we reset the counter.
        # We increase the counter if there is no improvement. Once the counter reaches the patience, we stop and remember the epoch.

        # Improvement means the loss decreased by more than min_delta
        if (self.best_loss - val_loss) > self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            # No meaningful improvement this epoch
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
                self.stop_epoch = epoch
        return self.stop

In [ ]:
# --- FULL evaluate_fold_TL (single learning rate) ---
import copy
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import root_mean_squared_error, r2_score
import pandas as pd
from NN_model import ImprovedNN

def evaluate_fold_TL(
    trial, fold_idx,
    X_train_scaled, y_train,
    X_val_scaled,   y_val,
    hidden_layers, dropout_rate,
    learning_rate, weight_decay, batch_size,
    freeze_level,                # 0,1,2,3 → how many feature blocks to freeze
    baseline_ckpt,               # path to medium-range baseline .pth
    max_epochs = 10**9,
    patience   = 30,
    min_delta  = 0.0,
    X_test_scaled=None, y_test=None,
    save_checkpoints=False, checkpoint_dir="checkpoints", save_every_n_epochs=15
):
    """
    Transfer-learning fold trainer using a SINGLE learning rate (no param groups).
    Expects pre-scaled numpy arrays (no scaling here).

    Returns:
        rmse, r2, q2, model, train_losses, val_losses, stop_epoch
    """
    device = torch.device("cpu")
    print(f"Fold {fold_idx}: TL on cpu | freeze={freeze_level} | lr={learning_rate:g}")

    # checkpoint bookkeeping
    checkpoint_tracking = []
    fold_checkpoint_dir = None
    if save_checkpoints:
        checkpoint_path = Path(checkpoint_dir)
        checkpoint_path.mkdir(parents=True, exist_ok=True)
        fold_checkpoint_dir = checkpoint_path / f"fold_{fold_idx}"
        fold_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        print(f"Checkpoints will be saved to: {fold_checkpoint_dir}")

    # tensors/loaders (inputs are already scaled)
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32, device=device)
    y_train_tensor = torch.tensor(y_train,        dtype=torch.float32, device=device)
    X_val_tensor   = torch.tensor(X_val_scaled,   dtype=torch.float32, device=device)
    y_val_tensor   = torch.tensor(y_val,          dtype=torch.float32, device=device)

    train_loader = DataLoader(
        TensorDataset(X_train_tensor, y_train_tensor),
        batch_size=batch_size, shuffle=True, num_workers=0
    )
    val_loader = DataLoader(
        TensorDataset(X_val_tensor, y_val_tensor),
        batch_size=batch_size, shuffle=False, num_workers=0
    )

    # --- model: same arch as baseline; load baseline weights ---
    model = ImprovedNN(
        input_size = X_train_scaled.shape[1],
        hidden_layers = hidden_layers,
        dropout_rate  = dropout_rate
    ).to(device)

    state = torch.load(baseline_ckpt, map_location=device)
    model.load_state_dict(state, strict=True)

    # --- freeze policy ---
    set_freeze_mode(model, freeze_level)

    # --- optimizer: SINGLE LR over all trainable params ---
    optimizer = optim.AdamW(
        (p for p in model.parameters() if p.requires_grad),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    # loss & scheduler & early stopping (same semantics as baseline)
    criterion = RMSELoss()  # your existing class
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    early_stopper = EarlyStopper(patience=patience, min_delta=min_delta)

    best_val_loss = float('inf')
    best_state = copy.deepcopy(model.state_dict())
    train_losses, val_losses = [], []
    stop_epoch = None

    # --- training loop ---
    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            preds = model(xb)                 # shape [B,] from your ImprovedNN
            loss  = criterion(preds, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                preds = model(xb)
                loss  = criterion(preds, yb)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        scheduler.step(val_loss)

        # track best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        # save periodic checkpoints
        if save_checkpoints and (epoch % save_every_n_epochs == 0 or epoch == 1):
            save_checkpoint(
                model, optimizer, epoch, train_loss, val_loss,
                y_train, y_val, val_loader, fold_idx,
                fold_checkpoint_dir, checkpoint_tracking, is_final=False
            )

        # early stopping
        if early_stopper.early_stop(val_loss, epoch=epoch):
            stop_epoch = early_stopper.stop_epoch
            print(f"[Fold {fold_idx}] Early stopping at epoch {stop_epoch} (best Val Loss: {best_val_loss:.4f})")
            if save_checkpoints and epoch % save_every_n_epochs != 0 and epoch != 1:
                save_checkpoint(
                    model, optimizer, epoch, train_loss, val_loss,
                    y_train, y_val, val_loader, fold_idx,
                    fold_checkpoint_dir, checkpoint_tracking, is_final=True
                )
            break

        if epoch % 50 == 0 or epoch == 1:
            print(f"[Fold {fold_idx}] Epoch {epoch:4d} | Train {train_loss:.4f} | Val {val_loss:.4f} | ES {early_stopper.counter}/{patience}")

    # restore best
    model.load_state_dict(best_state)
    model.eval()

    # optional: export the checkpoint-tracking spreadsheet
    if save_checkpoints and checkpoint_tracking:
        df_checkpoints = pd.DataFrame(checkpoint_tracking)
        spreadsheet_file = fold_checkpoint_dir / f"fold_{fold_idx}_checkpoints.csv"
        df_checkpoints.to_csv(spreadsheet_file, index=False)
        print(f"[Fold {fold_idx}] Checkpoint spreadsheet saved: {spreadsheet_file}")
        print(f"[Fold {fold_idx}] Total checkpoints saved: {len(checkpoint_tracking)}")

    # final metrics on validation
    all_preds = []
    with torch.no_grad():
        for xb, _ in val_loader:
            all_preds.append(model(xb).cpu().numpy())
    preds_val = np.concatenate(all_preds)

    rmse = root_mean_squared_error(y_val, preds_val)
    r2   = r2_score(y_val, preds_val)
    q2   = 1 - np.sum((y_val - preds_val)**2) / np.sum((y_val - y_train.mean())**2)

    return rmse, r2, q2, model, train_losses, val_losses, stop_epoch


In [ ]:
import numpy as np, pandas as pd, optuna, torch
from sklearn.model_selection import StratifiedKFold
from NN_model import ImprovedNN  # your model
from pathlib import Path
import json, torch

# 0) Load ALREADY-SCALED low-MW data (same feature order as baseline)
df_low = pd.read_csv("low_MW_scaled.csv")     # <- already scaled
TARGET_COL = "MP"

exclude = {"SMILES", TARGET_COL, "Structure_Cluster", "cv_fold", "Stratify_Label"}
num_cols = df_low.select_dtypes(include=[np.number]).columns
feature_cols = [c for c in num_cols if c not in exclude]

X = df_low[feature_cols].to_numpy(np.float32)  # <-- already scaled
y = df_low[TARGET_COL].to_numpy(np.float32)
y_strat = df_low["Structure_Cluster"].astype(str).to_numpy()

# 1) Fix folds once

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
folds = [(tr, va) for tr, va in skf.split(X, y_strat)]


BASELINE_CKPT = "artifacts_medium_cv/final_fold_models/fold_BEST_best.pth"  # Checkpoint or pipeline

# for TL runs (use a different root for baseline if you want)
TRIALS_OUT = Path("artifacts_low_TL_cv") / "trials"
TRIALS_OUT.mkdir(parents=True, exist_ok=True)



In [ ]:
from pathlib import Path
import json, joblib, numpy as np, pandas as pd, torch, optuna

# --- scenarios: name, vector (for your notes), freeze_level used by evaluate_fold_TL ---

HIDDEN_LAYERS = [256,128,64]   # must match baseline arch
N_TRIALS      = 30
OUT_ROOT      = Path("artifacts_low_TL_cv")   # common root for all scenarios
OUT_ROOT.mkdir(parents=True, exist_ok=True)

def run_one_scenario(tag, freeze_vec, freeze_level):
    print(f"\n=== Scenario: {tag} | freeze={freeze_vec} (level={freeze_level}) ===")
    SCEN_OUT = OUT_ROOT / tag
    (SCEN_OUT / "trials").mkdir(parents=True, exist_ok=True)

    def objective_tl_fixed(trial):
        # fixed freeze level; tune the rest
        learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
        weight_decay  = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
        batch_size    = trial.suggest_categorical("batch_size", [16, 32, 64])
        dropout_rate  = trial.suggest_float("dropout_rate", 0.2, 0.5)

        trial_dir = SCEN_OUT / "trials" / f"trial_{trial.number:04d}"
        trial_dir.mkdir(parents=True, exist_ok=True)

        fold_metrics, rmses = [], []
        for fold_idx, (tr_idx, va_idx) in enumerate(folds):
            X_tr, y_tr = X[tr_idx], y[tr_idx]
            X_va, y_va = X[va_idx], y[va_idx]

            rmse, r2, q2, model, *_ = evaluate_fold_TL(
                trial=trial,
                fold_idx=fold_idx,
                X_train_scaled=X_tr, y_train=y_tr,
                X_val_scaled=X_va,   y_val=y_va,
                hidden_layers=HIDDEN_LAYERS, dropout_rate=dropout_rate,
                learning_rate=learning_rate, weight_decay=weight_decay,
                batch_size=batch_size, freeze_level=freeze_level,
                baseline_ckpt=BASELINE_CKPT,
                max_epochs=10**6, patience=30, min_delta=0.0,
                save_checkpoints=False
            )

            ckpt_path = trial_dir / f"fold_{fold_idx}_best.pth"
            torch.save(model.state_dict(), ckpt_path)

            fold_metrics.append({
                "fold": fold_idx,
                "rmse": float(rmse),
                "r2":   float(r2),
                "q2":   float(q2),
                "checkpoint": str(ckpt_path)
            })
            rmses.append(rmse)

        summary = {
            "scenario": tag,
            "freeze_vector": freeze_vec,
            "freeze_level": freeze_level,
            "trial_number": trial.number,
            "params": {
                "learning_rate": learning_rate,
                "weight_decay":  weight_decay,
                "batch_size":    batch_size,
                "dropout_rate":  dropout_rate,
                "hidden_layers": HIDDEN_LAYERS
            },
            "avg_rmse": float(np.mean(rmses)),
            "folds":    fold_metrics
        }
        with open(trial_dir / "summary.json", "w") as f:
            json.dump(summary, f, indent=2)

        return float(np.mean(rmses))

    # -- HPO
    study = optuna.create_study(direction="minimize")
    study.optimize(objective_tl_fixed, n_trials=N_TRIALS, gc_after_trial=True)

    # save study artifacts
    joblib.dump(study, SCEN_OUT / "study.joblib")
    study.trials_dataframe().to_csv(SCEN_OUT / "trials.csv", index=False)
    with open(SCEN_OUT / "best_params.json","w") as f:
        json.dump(study.best_params, f, indent=2)
    with open(SCEN_OUT / "best_value.txt","w") as f:
        f.write(f"{study.best_value:.6f}\n")
    print(f"[{tag}] Best avg RMSE: {study.best_value:.4f}")
    print(f"[{tag}] Best params:  {study.best_params}")

    # -- Final per-fold retrain with best params (to produce clean fold models + metrics)
    best = study.best_params
    FINAL_DIR = SCEN_OUT / "final_fold_models"
    FINAL_DIR.mkdir(parents=True, exist_ok=True)

    rows = []
    for fold_idx, (tr_idx, va_idx) in enumerate(folds):
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        X_va, y_va = X[va_idx], y[va_idx]

        rmse, r2, q2, model, *_ = evaluate_fold_TL(
            trial=None,
            fold_idx=fold_idx,
            X_train_scaled=X_tr, y_train=y_tr,
            X_val_scaled=X_va,   y_val=y_va,
            hidden_layers=HIDDEN_LAYERS,
            dropout_rate=best["dropout_rate"],
            learning_rate=best["learning_rate"],
            weight_decay=best["weight_decay"],
            batch_size=best["batch_size"],
            freeze_level=freeze_level,
            baseline_ckpt=BASELINE_CKPT,
            max_epochs=10**6, patience=30, min_delta=0.0,
            save_checkpoints=False
        )

        ckpt = FINAL_DIR / f"fold_{fold_idx}_best.pth"
        torch.save(model.state_dict(), ckpt)
        rows.append({"fold": fold_idx, "rmse": float(rmse), "r2": float(r2), "q2": float(q2), "checkpoint": str(ckpt)})

    cv_df = pd.DataFrame(rows).sort_values("rmse").reset_index(drop=True)
    cv_df.to_csv(SCEN_OUT / "cv_final_metrics.csv", index=False)

    best_row = cv_df.iloc[0]
    manifest = {
        "scenario": tag,
        "freeze_vector": freeze_vec,
        "freeze_level": freeze_level,
        "best_fold": int(best_row["fold"]),
        "checkpoint": best_row["checkpoint"],
        "hidden_layers": HIDDEN_LAYERS,
        "best_params": best
    }
    with open(SCEN_OUT / "manifest.json","w") as f:
        json.dump(manifest, f, indent=2)

    print(f"[{tag}] Best fold: {manifest['best_fold']} → {manifest['checkpoint']}")
    return study, cv_df, manifest


# ---------- RUN ALL THREE ----------
SCENARIOS = [
    ("no_freeze",        [0,0,0], 0),
    ("freeze_fc1",       [1,0,0], 1),
    ("freeze_fc1_fc2",   [1,1,0], 2),
]

results = {}
for tag, vec, lvl in SCENARIOS:
    study, cv_df, manifest = run_one_scenario(tag, vec, lvl)
    results[tag] = {"best": study.best_value, "manifest": manifest}
print("\nSummary:", json.dumps(results, indent=2))


In [ ]:
# Do the same thing for High range MP